# Identity Check

This notebook verifies the analytic upper-bound certificate directly on the full sequential+$S_3$ affine slice.

The proof target is

$$
F_{\mathrm{ave}}-\sum_{k=0}^8 \operatorname{Tr}(W_k H_k)
=
\operatorname{Tr}[(Y_0-W_0)H_0]
+
\operatorname{Tr}[(Y_1-W_1)H_1].
$$

Once this identity holds on the full affine slice, the bound follows from

$$
Y_0-W_0\succeq 0,\qquad Y_1-W_1\succeq 0,\qquad H_k\succeq 0.
$$


## Inputs

The only external data file used at runtime is `Omega_Tilde_Symbolic_d2_N3.pkl`.

Put that pickle file in the same folder as this notebook, and run the notebook from that folder. The 171 affine constraints are embedded below, so no other local notebook or helper script is needed.


In [1]:
from pathlib import Path
import pickle

import sympy as sp

RADICALS = [sp.sqrt(2), sp.sqrt(3), sp.sqrt(6)]
OMEGA_FILENAME = "Omega_Tilde_Symbolic_d2_N3.pkl"
OMEGA_PICKLE = Path(OMEGA_FILENAME)

if not OMEGA_PICKLE.exists():
    raise FileNotFoundError(
        f"Expected {OMEGA_FILENAME} in the same folder as this notebook. "
        "Run the notebook from that folder, or move the pickle next to it."
    )

# Each pair is (reduced block dimension, multiplicity).
BLOCKS_INFO = [
    (4, 1),
    (6, 3),
    (2, 5),
    (6, 3),
    (9, 9),
    (3, 15),
    (2, 5),
    (3, 15),
    (1, 25),
]




## Embedded Full Affine Constraints

The next code cell is written in the same shape as the printed output from `constraints.ipynb`: a pasteable `apply_derived_constraints(...)` block. The verification code extracts the `constraints.append(...)` lines from that block.


In [2]:
DERIVED_CONSTRAINTS_TEXT = r"""
def apply_derived_constraints(H0, H1, H2, H3, H4, H5, H6, H7, H8):
    constraints = []
    constraints.append(cp.imag(H0[0,1]) == 0)
    constraints.append(cp.imag(H0[0,2]) == 0)
    constraints.append(cp.imag(H0[0,3]) == 0)
    constraints.append(cp.imag(H0[1,2]) == 0)
    constraints.append(cp.imag(H0[1,3]) == 0)
    constraints.append(cp.imag(H0[2,3]) == 0)
    constraints.append(cp.real(H0[0,1]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,2]) == 3*sqrt(3)*cp.real(H1[3,3])/2 - 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,3]) == 3*cp.real(H1[1,3]) - 3*cp.real(H1[3,3]) + 3*cp.real(H1[4,4]))
    constraints.append(cp.real(H0[1,2]) == -3*cp.real(H1[1,3]))
    constraints.append(cp.real(H0[1,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[2,3]) == -3*sqrt(3)*cp.real(H1[3,3])/2 + 3*sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H0[0,0]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[1,1]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[2,2]) == -3*cp.real(H1[3,3]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.real(H0[3,3]) == -3*cp.real(H1[4,4]) - 27*cp.real(H4[5,5])/2 + 27*cp.real(H4[8,8])/4 - 45*cp.real(H5[1,1])/2 + 45*cp.real(H5[2,2])/4 - 15*cp.real(H7[2,2])/4 - 25*cp.real(H8[0,0])/4 + 2)
    constraints.append(cp.imag(H1[0,1]) == 0)
    constraints.append(cp.imag(H1[0,2]) == -sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[0,3]) == 0)
    constraints.append(cp.imag(H1[0,4]) == 0)
    constraints.append(cp.imag(H1[0,5]) == cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[1,2]) == -cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[1,3]) == 0)
    constraints.append(cp.imag(H1[1,4]) == 0)
    constraints.append(cp.imag(H1[1,5]) == -sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,3]) == -cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,4]) == sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.imag(H1[2,5]) == 0)
    constraints.append(cp.imag(H1[3,4]) == 0)
    constraints.append(cp.imag(H1[3,5]) == sqrt(3)*cp.imag(H1[4,5]))
    constraints.append(cp.real(H1[0,1]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[0,2]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[0,3]) == -sqrt(3)*cp.real(H1[3,3])/2 + sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[0,4]) == -cp.real(H1[1,3]) + cp.real(H1[3,3]) - cp.real(H1[4,4]))
    constraints.append(cp.real(H1[0,5]) == cp.real(H1[4,5]))
    constraints.append(cp.real(H1[1,2]) == -cp.real(H1[4,5]))
    constraints.append(cp.real(H1[1,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[1,5]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,3]) == cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,4]) == -sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[2,5]) == 0)
    constraints.append(cp.real(H1[3,4]) == sqrt(3)*cp.real(H1[3,3])/2 - sqrt(3)*cp.real(H1[4,4])/2)
    constraints.append(cp.real(H1[3,5]) == sqrt(3)*cp.real(H1[4,5]))
    constraints.append(cp.real(H1[0,0]) == cp.real(H1[4,4]))
    constraints.append(cp.real(H1[1,1]) == cp.real(H1[3,3]))
    constraints.append(cp.real(H1[2,2]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
    constraints.append(cp.real(H1[5,5]) == -5*cp.real(H2[1,1])/3 - 9*cp.real(H4[5,5]) + 9*cp.real(H4[8,8])/2 - 15*cp.real(H5[1,1]) + 15*cp.real(H5[2,2])/2 - 5*cp.real(H7[2,2])/2 - 25*cp.real(H8[0,0])/6 + 4/3)
    constraints.append(cp.imag(H2[0,1]) == 0)
    constraints.append(cp.real(H2[0,1]) == 0)
    constraints.append(cp.real(H2[0,0]) == cp.real(H2[1,1]))
    constraints.append(cp.imag(H3[0,1]) == 0)
    constraints.append(cp.imag(H3[0,2]) == 0)
    constraints.append(cp.imag(H3[0,3]) == 4*sqrt(2)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[0,4]) == 3*sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[0,5]) == cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,2]) == 4*sqrt(2)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,3]) == 0)
    constraints.append(cp.imag(H3[1,4]) == cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[1,5]) == -3*sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[2,3]) == 0)
    constraints.append(cp.imag(H3[2,4]) == 3*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[2,5]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[3,4]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[3,5]) == -3*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H3[4,5]) == 0)
    constraints.append(cp.real(H3[0,1]) == 3*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[6,6])/2 - 3*sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H3[0,2]) == -3*sqrt(3)*cp.real(H4[3,3])/2 - 9*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 15*sqrt(3)*cp.real(H4[6,6])/4 + 9*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[0,3]) == sqrt(6)*cp.real(H4[4,6])/2 - 3*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 69*cp.real(H4[6,6])/32 + 69*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H3[0,4]) == -3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 15*sqrt(6)*cp.real(H4[6,6])/32 + 81*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[0,5]) == -sqrt(3)*cp.real(H4[4,6]) + 3*sqrt(2)*cp.real(H4[3,3]) - 3*sqrt(2)*cp.real(H4[4,4]) + 21*sqrt(2)*cp.real(H4[6,6])/16 - 21*sqrt(2)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H3[1,2]) == -sqrt(6)*cp.real(H4[4,6])/2 - 27*cp.real(H4[6,6])/32 + 27*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H3[1,3]) == -9*sqrt(3)*cp.real(H4[3,3])/2 - 3*sqrt(3)*cp.real(H4[4,4])/2 + 3*sqrt(3)*cp.real(H4[5,5]) + 9*sqrt(3)*cp.real(H4[6,6])/4 + 15*sqrt(3)*cp.real(H4[7,7])/4 - 3*sqrt(3)*cp.real(H4[8,8]) + 5*sqrt(3)*cp.real(H5[1,1]) - 5*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[1,4]) == sqrt(3)*cp.real(H4[4,6]))
    constraints.append(cp.real(H3[1,5]) == -3*sqrt(6)*cp.real(H4[4,4]) + 3*sqrt(6)*cp.real(H4[5,5])/2 + 81*sqrt(6)*cp.real(H4[6,6])/32 + 15*sqrt(6)*cp.real(H4[7,7])/32 - 3*sqrt(6)*cp.real(H4[8,8])/2 + 5*sqrt(6)*cp.real(H5[1,1])/2 - 5*sqrt(6)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[2,3]) == -3*sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/2 + 3*sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H3[2,4]) == -3*sqrt(2)*cp.real(H4[3,3]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 15*sqrt(2)*cp.real(H4[6,6])/32 + 81*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[2,5]) == 3*cp.real(H4[4,6]) - 3*sqrt(6)*cp.real(H4[3,3]) + 3*sqrt(6)*cp.real(H4[4,4]) - 21*sqrt(6)*cp.real(H4[6,6])/16 + 21*sqrt(6)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H3[3,4]) == -3*cp.real(H4[4,6]))
    constraints.append(cp.real(H3[3,5]) == -3*sqrt(2)*cp.real(H4[4,4]) + 3*sqrt(2)*cp.real(H4[5,5])/2 + 81*sqrt(2)*cp.real(H4[6,6])/32 + 15*sqrt(2)*cp.real(H4[7,7])/32 - 3*sqrt(2)*cp.real(H4[8,8])/2 + 5*sqrt(2)*cp.real(H5[1,1])/2 - 5*sqrt(2)*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[4,5]) == 0)
    constraints.append(cp.real(H3[0,0]) == -6*cp.real(H4[3,3]) - 9*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 15*cp.real(H4[6,6])/2 + 9*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[1,1]) == -9*cp.real(H4[3,3]) - 6*cp.real(H4[4,4]) + 15*cp.real(H4[5,5])/2 + 9*cp.real(H4[6,6])/2 + 15*cp.real(H4[7,7])/2 - 6*cp.real(H4[8,8]) + 25*cp.real(H5[1,1])/2 - 10*cp.real(H5[2,2]))
    constraints.append(cp.real(H3[2,2]) == -3*cp.real(H4[3,3]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
    constraints.append(cp.real(H3[3,3]) == -3*cp.real(H4[4,4]) + 3*cp.real(H4[5,5])/2 + 5*cp.real(H5[1,1])/2)
    constraints.append(cp.real(H3[4,4]) == -3*cp.real(H4[6,6]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
    constraints.append(cp.real(H3[5,5]) == -3*cp.real(H4[7,7]) + 3*cp.real(H4[8,8])/2 + 5*cp.real(H5[2,2])/2)
    constraints.append(cp.imag(H4[0,1]) == 0)
    constraints.append(cp.imag(H4[0,2]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8 - sqrt(3)*cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[0,3]) == 0)
    constraints.append(cp.imag(H4[0,4]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[0,5]) == -11*sqrt(6)*cp.imag(H4[5,6])/24 + 7*sqrt(2)*cp.imag(H4[5,7])/8 + cp.imag(H4[7,8]))
    constraints.append(cp.imag(H4[0,6]) == -sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[0,7]) == -cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[0,8]) == sqrt(3)*cp.imag(H4[5,6])/6 + cp.imag(H4[5,7])/2 + sqrt(2)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[1,2]) == sqrt(6)*cp.imag(H4[5,6])/8 + sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[1,3]) == -4*sqrt(2)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[1,4]) == 0)
    constraints.append(cp.imag(H4[1,5]) == 3*sqrt(2)*cp.imag(H4[5,6])/8 + sqrt(6)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[1,6]) == -cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[1,7]) == sqrt(3)*cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[1,8]) == -3*cp.imag(H4[5,6])/2 + sqrt(3)*cp.imag(H4[5,7])/2 + 3*sqrt(6)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[2,3]) == -5*sqrt(6)*cp.imag(H4[5,6])/24 + 9*sqrt(2)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[2,4]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8)
    constraints.append(cp.imag(H4[2,5]) == 0)
    constraints.append(cp.imag(H4[2,6]) == -sqrt(3)*cp.imag(H4[5,6])/3)
    constraints.append(cp.imag(H4[2,7]) == sqrt(3)*cp.imag(H4[5,7]))
    constraints.append(cp.imag(H4[2,8]) == 0)
    constraints.append(cp.imag(H4[3,4]) == 0)
    constraints.append(cp.imag(H4[3,5]) == -3*sqrt(2)*cp.imag(H4[5,6])/8 - sqrt(6)*cp.imag(H4[5,7])/8 + sqrt(3)*cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[3,6]) == -cp.imag(H4[4,7]))
    constraints.append(cp.imag(H4[3,7]) == sqrt(3)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[3,8]) == -cp.imag(H4[5,6])/2 - sqrt(3)*cp.imag(H4[5,7])/2 - sqrt(6)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[4,5]) == -sqrt(6)*cp.imag(H4[5,6])/8 - sqrt(2)*cp.imag(H4[5,7])/8 - cp.imag(H4[7,8])/2)
    constraints.append(cp.imag(H4[4,6]) == sqrt(3)*cp.imag(H4[4,7])/3)
    constraints.append(cp.imag(H4[4,8]) == -sqrt(3)*cp.imag(H4[5,6])/2 + cp.imag(H4[5,7])/2 + 3*sqrt(2)*cp.imag(H4[7,8])/8)
    constraints.append(cp.imag(H4[5,8]) == 0)
    constraints.append(cp.imag(H4[6,7]) == 0)
    constraints.append(cp.imag(H4[6,8]) == 0)
    constraints.append(cp.real(H4[0,1]) == -sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - sqrt(3)*cp.real(H4[6,6])/2 + sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[0,2]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8 - sqrt(3)*cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[0,3]) == sqrt(3)*cp.real(H4[3,3])/2 + 3*sqrt(3)*cp.real(H4[4,4])/2 - 5*sqrt(3)*cp.real(H4[6,6])/4 - 3*sqrt(3)*cp.real(H4[7,7])/4)
    constraints.append(cp.real(H4[0,4]) == -sqrt(6)*cp.real(H4[4,6])/6 + cp.real(H4[3,3]) - cp.real(H4[4,4]) + 23*cp.real(H4[6,6])/32 - 23*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[0,5]) == 11*sqrt(6)*cp.real(H4[5,6])/24 - 7*sqrt(2)*cp.real(H4[5,7])/8 + cp.real(H4[7,8]))
    constraints.append(cp.real(H4[0,6]) == sqrt(6)*cp.real(H4[3,3]) - 5*sqrt(6)*cp.real(H4[6,6])/32 - 27*sqrt(6)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[0,7]) == sqrt(3)*cp.real(H4[4,6])/3 - sqrt(2)*cp.real(H4[3,3]) + sqrt(2)*cp.real(H4[4,4]) - 7*sqrt(2)*cp.real(H4[6,6])/16 + 7*sqrt(2)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H4[0,8]) == -sqrt(3)*cp.real(H4[5,6])/6 - cp.real(H4[5,7])/2 + sqrt(2)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[1,2]) == -sqrt(6)*cp.real(H4[5,6])/8 - sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[1,3]) == sqrt(6)*cp.real(H4[4,6])/6 + 9*cp.real(H4[6,6])/32 - 9*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[1,4]) == 3*sqrt(3)*cp.real(H4[3,3])/2 + sqrt(3)*cp.real(H4[4,4])/2 - 3*sqrt(3)*cp.real(H4[6,6])/4 - 5*sqrt(3)*cp.real(H4[7,7])/4)
    constraints.append(cp.real(H4[1,5]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[1,6]) == -sqrt(3)*cp.real(H4[4,6])/3)
    constraints.append(cp.real(H4[1,7]) == sqrt(6)*cp.real(H4[4,4]) - 27*sqrt(6)*cp.real(H4[6,6])/32 - 5*sqrt(6)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[1,8]) == 3*cp.real(H4[5,6])/2 - sqrt(3)*cp.real(H4[5,7])/2 + 3*sqrt(6)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[2,3]) == -5*sqrt(6)*cp.real(H4[5,6])/24 + 9*sqrt(2)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[2,4]) == -3*sqrt(2)*cp.real(H4[5,6])/8 - sqrt(6)*cp.real(H4[5,7])/8)
    constraints.append(cp.real(H4[2,5]) == 2*sqrt(3)*cp.real(H4[5,5]) - 2*sqrt(3)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[2,6]) == -sqrt(3)*cp.real(H4[5,6])/3)
    constraints.append(cp.real(H4[2,7]) == sqrt(3)*cp.real(H4[5,7]))
    constraints.append(cp.real(H4[2,8]) == sqrt(6)*cp.real(H4[5,5]) - sqrt(6)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[3,4]) == sqrt(3)*cp.real(H4[3,3])/2 - sqrt(3)*cp.real(H4[4,4])/2 + sqrt(3)*cp.real(H4[6,6])/2 - sqrt(3)*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[3,5]) == 3*sqrt(2)*cp.real(H4[5,6])/8 + sqrt(6)*cp.real(H4[5,7])/8 + sqrt(3)*cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[3,6]) == sqrt(2)*cp.real(H4[3,3]) - 5*sqrt(2)*cp.real(H4[6,6])/32 - 27*sqrt(2)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[3,7]) == -cp.real(H4[4,6]) + sqrt(6)*cp.real(H4[3,3]) - sqrt(6)*cp.real(H4[4,4]) + 7*sqrt(6)*cp.real(H4[6,6])/16 - 7*sqrt(6)*cp.real(H4[7,7])/16)
    constraints.append(cp.real(H4[3,8]) == cp.real(H4[5,6])/2 + sqrt(3)*cp.real(H4[5,7])/2 - sqrt(6)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[4,5]) == sqrt(6)*cp.real(H4[5,6])/8 + sqrt(2)*cp.real(H4[5,7])/8 - cp.real(H4[7,8])/2)
    constraints.append(cp.real(H4[4,7]) == sqrt(2)*cp.real(H4[4,4]) - 27*sqrt(2)*cp.real(H4[6,6])/32 - 5*sqrt(2)*cp.real(H4[7,7])/32)
    constraints.append(cp.real(H4[4,8]) == sqrt(3)*cp.real(H4[5,6])/2 - cp.real(H4[5,7])/2 + 3*sqrt(2)*cp.real(H4[7,8])/8)
    constraints.append(cp.real(H4[5,8]) == sqrt(2)*cp.real(H4[5,5]) - sqrt(2)*cp.real(H4[8,8]))
    constraints.append(cp.real(H4[6,7]) == 0)
    constraints.append(cp.real(H4[6,8]) == 0)
    constraints.append(cp.real(H4[0,0]) == 2*cp.real(H4[3,3]) + 3*cp.real(H4[4,4]) - 5*cp.real(H4[6,6])/2 - 3*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[1,1]) == 3*cp.real(H4[3,3]) + 2*cp.real(H4[4,4]) - 3*cp.real(H4[6,6])/2 - 5*cp.real(H4[7,7])/2)
    constraints.append(cp.real(H4[2,2]) == 5*cp.real(H4[5,5]) - 4*cp.real(H4[8,8]))
    constraints.append(cp.imag(H5[0,1]) == 0)
    constraints.append(cp.imag(H5[0,2]) == 0)
    constraints.append(cp.imag(H5[1,2]) == 0)
    constraints.append(cp.real(H5[0,1]) == 2*sqrt(3)*cp.real(H5[1,1]) - 2*sqrt(3)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[0,2]) == sqrt(6)*cp.real(H5[1,1]) - sqrt(6)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[1,2]) == sqrt(2)*cp.real(H5[1,1]) - sqrt(2)*cp.real(H5[2,2]))
    constraints.append(cp.real(H5[0,0]) == 5*cp.real(H5[1,1]) - 4*cp.real(H5[2,2]))
    constraints.append(cp.imag(H6[0,1]) == 0)
    constraints.append(cp.real(H6[0,1]) == 0)
    constraints.append(cp.real(H6[0,0]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
    constraints.append(cp.real(H6[1,1]) == -3*cp.real(H7[1,1]) + 3*cp.real(H7[2,2])/2 + 5*cp.real(H8[0,0])/2)
    constraints.append(cp.imag(H7[0,1]) == 0)
    constraints.append(cp.imag(H7[0,2]) == 0)
    constraints.append(cp.imag(H7[1,2]) == 0)
    constraints.append(cp.real(H7[0,1]) == 0)
    constraints.append(cp.real(H7[0,2]) == 0)
    constraints.append(cp.real(H7[1,2]) == 0)
    constraints.append(cp.real(H7[0,0]) == cp.real(H7[1,1]))
    return constraints
"""

CONSTRAINT_LINES = []
for line in DERIVED_CONSTRAINTS_TEXT.splitlines():
    stripped = line.strip()
    if stripped.startswith("constraints.append("):
        CONSTRAINT_LINES.append(stripped)

print("embedded constraints:", len(CONSTRAINT_LINES))


embedded constraints: 171


## Turn Constraints Into a Linear System

Each block `H_k` is represented as a symbolic Hermitian matrix. Evaluating the constraint lines on those matrices gives linear equations in the scalar entries of all blocks.


In [3]:
class SimpleCP:
    @staticmethod
    def real(expr):
        return sp.simplify(sp.re(expr))

    @staticmethod
    def imag(expr):
        return sp.simplify(sp.im(expr))


def make_hermitian_blocks():
    blocks = []
    variables = []

    for block_index, (dim, _multiplicity) in enumerate(BLOCKS_INFO):
        matrix = sp.zeros(dim, dim)

        for row in range(dim):
            real_symbol = sp.Symbol(f"H{block_index}R_{row}{row}", real=True)
            matrix[row, row] = real_symbol
            variables.append(real_symbol)

            for col in range(row + 1, dim):
                real_symbol = sp.Symbol(f"H{block_index}R_{row}{col}", real=True)
                imag_symbol = sp.Symbol(f"H{block_index}I_{row}{col}", real=True)
                matrix[row, col] = real_symbol + sp.I * imag_symbol
                matrix[col, row] = real_symbol - sp.I * imag_symbol
                variables.append(real_symbol)
                variables.append(imag_symbol)

        blocks.append(matrix)

    return blocks, variables


def constraint_to_expression(line, blocks):
    prefix = "constraints.append("
    inner = line[len(prefix):-1]
    lhs_code, rhs_code = inner.split("==", 1)

    env = {"cp": SimpleCP, "sqrt": sp.sqrt, "sp": sp}
    for index, block in enumerate(blocks):
        env[f"H{index}"] = block

    lhs = eval(lhs_code, env)
    rhs = eval(rhs_code, env)
    expression = sp.re(sp.expand(lhs - rhs))
    return sp.simplify(sp.nsimplify(expression, RADICALS))


def build_linear_system():
    blocks, variables = make_hermitian_blocks()
    equations = []

    for line in CONSTRAINT_LINES:
        equations.append(constraint_to_expression(line, blocks))

    A, b = sp.linear_eq_to_matrix(equations, variables)
    return A, b, blocks, variables


## Solve the Affine Slice

The linear equations define an affine space of feasible block entries. We solve the equations once and substitute the result back into the symbolic blocks.


In [4]:
A, b, raw_blocks, variables = build_linear_system()
solution = list(sp.linsolve((A, b), variables))[0]
substitutions = dict(zip(variables, solution))

free_variables = []
for symbol, expression in zip(variables, solution):
    if expression == symbol:
        free_variables.append(symbol)

blocks = []
for raw_block in raw_blocks:
    block = raw_block.applyfunc(lambda expr: sp.simplify(expr.subs(substitutions)))
    blocks.append(block)

print("scalar variables:", len(variables))
print("constraints:", len(CONSTRAINT_LINES))
print("rank:", A.rank())
print("augmented rank:", A.row_join(b).rank())
print("free variables:", len(free_variables))


scalar variables: 196
constraints: 171
rank: 171
augmented rank: 171
free variables: 25


## Build the Weighted Objective Blocks

The performance operator is block diagonal after the Schur transform. For a block with multiplicity `d_k`, the primal objective uses

$$
W_k=d_k\Omega_k.
$$

The code below obtains `W_k` by summing the diagonal multiplicity sectors of the stored operator.


In [5]:
def load_weighted_objective_blocks():
    with OMEGA_PICKLE.open("rb") as handle:
        data = pickle.load(handle)

    omega = data["Omega_tilde"]
    p_symbol = data["p_symbol"]
    weighted_blocks = []
    offset = 0

    for dim, multiplicity in BLOCKS_INFO:
        raw_block = omega[offset:offset + dim * multiplicity, offset:offset + dim * multiplicity]
        weighted_block = sp.zeros(dim, dim)

        for sector in range(multiplicity):
            lo = sector * dim
            hi = lo + dim
            weighted_block += raw_block[lo:hi, lo:hi]

        weighted_block = weighted_block.applyfunc(lambda expr: sp.simplify(sp.nsimplify(sp.re(expr), RADICALS)))
        weighted_blocks.append(weighted_block)
        offset += dim * multiplicity

    return p_symbol, weighted_blocks


p, W = load_weighted_objective_blocks()
print("symbol:", p)
print("block shapes:", [matrix.shape for matrix in W])


symbol: p
block shapes: [(4, 4), (6, 6), (2, 2), (6, 6), (9, 9), (3, 3), (2, 2), (3, 3), (1, 1)]


## Analytic Certificate

The candidate certificate only needs the two blocks `Y_0` and `Y_1`. The expected value is

$$
F_{\mathrm{ave}}=\frac{4}{3}(y_1+y_4+2y_2).
$$


In [6]:
def analytic_y_values(p):
    s2 = sp.sqrt(2)
    s3 = sp.sqrt(3)

    y1 = (
        -sp.Rational(25, 128) * p**3
        + s2 * p**3 / 16
        - 3 * s2 * p**2 / 16
        + sp.Rational(105, 128) * p**2
        - sp.Rational(73, 64) * p
        + s2 * p / 8
        + sp.Rational(9, 16)
    )
    y2 = p * (5 * p**2 + 4 * s2 * p**2 - 18 * p - 12 * s2 * p + 8 * s2 + 16) / 64
    y4 = (
        -sp.Rational(7, 128) * p**3
        + s2 * p**3 / 16
        - 3 * s2 * p**2 / 16
        + sp.Rational(27, 128) * p**2
        - sp.Rational(19, 64) * p
        + s2 * p / 8
        + sp.Rational(3, 16)
    )
    y6 = 3 * s3 * (3 * p**3 - 13 * p**2 + 18 * p - 8) / 128
    y7 = (
        s2 * p**3 / 16
        + sp.Rational(29, 128) * p**3
        - sp.Rational(75, 128) * p**2
        - 3 * s2 * p**2 / 16
        + s2 * p / 8
        + sp.Rational(35, 64) * p
        - sp.Rational(3, 16)
    )
    y8 = (
        -sp.Rational(47, 128) * p**3
        - s2 * p**3 / 16
        + 3 * s2 * p**2 / 16
        + sp.Rational(153, 128) * p**2
        - sp.Rational(89, 64) * p
        - s2 * p / 8
        + sp.Rational(9, 16)
    )

    return [sp.factor(value) for value in [y1, y2, y4, y6, y7, y8]]


def certificate_blocks(p):
    y1, y2, y4, y6, y7, y8 = analytic_y_values(p)

    Y0 = sp.Matrix([
        [y4, y6, y6, y7],
        [y6, y1, y8, -y6],
        [y6, y8, y1, -y6],
        [y7, -y6, -y6, y4],
    ]) / 3

    Y1 = sp.Matrix([
        [y4, y6, 0, y6, y7, 0],
        [y6, y1, 0, y8, -y6, 0],
        [0, 0, y2, 0, 0, 0],
        [y6, y8, 0, y1, -y6, 0],
        [y7, -y6, 0, -y6, y4, 0],
        [0, 0, 0, 0, 0, y2],
    ])

    return Y0, Y1


def expected_value_with_prefactor(prefactor, p):
    y1, y2, y4, _y6, _y7, _y8 = analytic_y_values(p)
    return sp.factor(prefactor * (y1 + y4 + 2 * y2))


def expected_value(p):
    return expected_value_with_prefactor(sp.Rational(4, 3), p)


Y0, Y1 = certificate_blocks(p)
Q0 = (Y0 - W[0]).applyfunc(sp.simplify)
Q1 = (Y1 - W[1]).applyfunc(sp.simplify)
F_ave = expected_value(p)

print("F_ave =", F_ave)


F_ave = (p - 2)*(-3*p**2 + 8*sqrt(2)*p**2 - 8*sqrt(2)*p + 9*p - 12)/24


## Check Positive Semidefiniteness

The slack matrices are `Q_0=Y_0-W_0` and `Q_1=Y_1-W_1`. Their eigenvalues are nonnegative for `0 <= p <= 1`.


In [7]:
def print_eigenvalues(name, matrix):
    print(name)
    for eigenvalue, multiplicity in matrix.copy().eigenvals().items():
        print("  multiplicity", multiplicity, ":", sp.factor(eigenvalue))


print_eigenvalues("Q0 = Y0 - W0", Q0)
print_eigenvalues("Q1 = Y1 - W1", Q1)


Q0 = Y0 - W0
  multiplicity 1 : -p**2*(p - 1)/16
  multiplicity 2 : p*(p - 1)*(2*sqrt(2)*p + 5*p - 4*sqrt(2) - 4)/48
  multiplicity 1 : 0
Q1 = Y1 - W1
  multiplicity 1 : -(p - 1)*(13*p**2 - 32*p + 24)/16
  multiplicity 2 : 3*sqrt(2)*p*(p - 2)*(p - 1)/16
  multiplicity 3 : 0


## Check the Remainder Identity

Now substitute the full affine solution into

$$
R = F_{\mathrm{ave}}-\sum_{k=0}^8\operatorname{Tr}(W_kH_k)
-\operatorname{Tr}(Q_0H_0)-\operatorname{Tr}(Q_1H_1).
$$

Since the affine solution is linear in the free variables, `R` is zero on the whole affine slice exactly when its constant term and all free-variable coefficients are zero.


In [8]:
def trace_inner(A, H):
    return sp.expand(sp.trace(A * H))


def simplify_exact(expr):
    expr = sp.simplify(expr)
    expr = sp.nsimplify(expr, RADICALS, tolerance=sp.Rational(1, 10) ** 12)
    return sp.factor(sp.simplify(expr))


objective = 0
for k in range(len(W)):
    objective += trace_inner(W[k], blocks[k])


def nonzero_p_coefficients(expr):
    expr = simplify_exact(sp.expand(expr))
    if expr == 0:
        return []

    poly = sp.Poly(sp.expand(expr), p)
    nonzero_terms = []
    for monomial, coefficient in poly.terms():
        coefficient = simplify_exact(coefficient)
        if coefficient != 0:
            nonzero_terms.append((monomial[0], coefficient))
    return nonzero_terms


def check_remainder_identity(F_candidate, assert_identity=True):
    remainder = sp.expand(
        F_candidate
        - objective
        - trace_inner(Q0, blocks[0])
        - trace_inner(Q1, blocks[1])
    )
    remainder = simplify_exact(remainder)

    constant_part = remainder
    for symbol in free_variables:
        constant_part = constant_part.subs(symbol, 0)
    constant_part = simplify_exact(constant_part)

    nonzero_coefficients = []
    for symbol in free_variables:
        coefficient = simplify_exact(remainder.coeff(symbol))
        if coefficient != 0:
            nonzero_coefficients.append((symbol, coefficient))

    p_coefficient_failures = []
    for label, expression in [("constant", constant_part)]:
        nonzero_terms = nonzero_p_coefficients(expression)
        if nonzero_terms:
            p_coefficient_failures.append((label, nonzero_terms))

    for symbol in free_variables:
        expression = simplify_exact(remainder.coeff(symbol))
        nonzero_terms = nonzero_p_coefficients(expression)
        if nonzero_terms:
            p_coefficient_failures.append((symbol, nonzero_terms))

    result = {
        "remainder": remainder,
        "constant_part": constant_part,
        "nonzero_coefficients": nonzero_coefficients,
        "p_coefficient_failures": p_coefficient_failures,
    }

    if assert_identity:
        assert constant_part == 0
        assert nonzero_coefficients == []
        assert p_coefficient_failures == []

    return result


identity_check = check_remainder_identity(F_ave, assert_identity=True)

print("free variables:", len(free_variables))
print("remainder constant:", identity_check["constant_part"])
print("nonzero coefficient count:", len(identity_check["nonzero_coefficients"]))
print("nonzero p-coefficient groups:", len(identity_check["p_coefficient_failures"]))


free variables: 25
remainder constant: 0
nonzero coefficient count: 0
nonzero p-coefficient groups: 0


## Conclusion

The symbolic check proves the identity on the full 171-constraint affine slice. Therefore every feasible strategy satisfies

$$
\sum_{k=0}^8 \operatorname{Tr}(W_kH_k)\le F_{\mathrm{ave}}.
$$
